# Spacing Statistics - RingEnergy

## 1. Importing / Installing Packages

In [29]:
from __future__ import annotations
from typing import List, Optional, Dict

import os # Importing os module for operating system dependent functionality

import glob # Importing glob module for file pattern matching

import pandas as pd # Importing pandas package

import datetime # Importing datetime module for date and time manipulation

# Set the maximum number of columns to display to None
pd.set_option('display.max_columns', None)

from pathlib import Path # Importing Path class from pathlib for filesystem path manipulation

from src.utils import reorder_columns, clean_column_names # Importing utility functions from the utils module

from src.well_data import GeoSurveyProcessor, WellDataLoader, WellSpacingCalculator, DirectionalBenchNeighbors, debug_pair_spacing # Importing custom classes for well data processing

from src.utils import DatabricksOdbcConnector # Importing DatabricksOdbcConnector class from utils module

## 2. Importing Data to Dataframes

In [2]:
def load_uwi_files(folder_path: str) -> pd.DataFrame:
    """
    Reads all .UWI files in the given folder into a single DataFrame.
    
    Parameters
    ----------
    folder_path : str
        Path to the folder containing .UWI files.
    
    Returns
    -------
    pd.DataFrame
        DataFrame with columns ['UWI', 'county'].
    """
    all_rows = []

    # Find all .UWI files in the folder
    uwi_paths = glob.glob(os.path.join(folder_path, "*.UWI"))

    for path in uwi_paths:
        county = Path(path).stem  # Extract filename without extension
        with open(path, "r", encoding="utf-8", errors="ignore") as f:
            uwis = [line.strip() for line in f if line.strip()]

        print(f"{county}: {len(uwis)} rows")

        for uwi in uwis:
            all_rows.append({"uwi": uwi, "county": county})

    df = pd.DataFrame(all_rows, columns=["uwi", "county"])
    return df

### 2.1 Importing header

In [3]:
data_loader = WellDataLoader(
    db = DatabricksOdbcConnector(),
    log_dir=r"C:\Users\Apoorva.Saxena\OneDrive - Sitio Royalties\Desktop\Project - Apoorva\Python\Parent_Child_Spacing\logs"
    )

In [4]:
df_header = data_loader.get_header_data(
    source=r"C:\Users\Apoorva.Saxena\OneDrive - Sitio Royalties\Desktop\Project - Apoorva\For Matt\Well Header\uwi_by_county_bench.xlsx",
    column_map={
        "uwi": "uwi",
        "api10":"api10",
        "well_name": "WellName",
        "operator": "Operator",
        "rsv_cat": "RsvCat",
        "bench": "bench",
        "first_prod_date": "FirstProd_Novi",
        "hole_direction": "HoleDirection_Final",
        "ihs_missing": "IHS_Missing",
        "ihs_ds_missing": "IHS_DS_Missing"
    },
    dtype={"uwi": str, "api10": str}
)

[WellDataLoaderLogger] INFO (08-21 09:57 PM): Loading header data from file: C:\Users\Apoorva.Saxena\OneDrive - Sitio Royalties\Desktop\Project - Apoorva\For Matt\Well Header\uwi_by_county_bench.xlsx (Line: 242) [well_data_manager.py]



In [5]:
df_header.shape

(2698, 10)

In [6]:
# Convert 'first_prod_date' to datetime, handling time values as NaT
mask = df_header['first_prod_date'].apply(lambda x: isinstance(x, datetime.time))
df_header.loc[mask, 'first_prod_date'] = pd.NaT
df_header['first_prod_date'] = pd.to_datetime(df_header['first_prod_date'], errors='coerce')

### 2.2 Getting Directional Surveys

In [7]:
# df_directional = data_loader.get_directional_data()

df_directional = pd.read_csv(
    r"C:\Users\Apoorva.Saxena\OneDrive - Sitio Royalties\Desktop\Project - Apoorva\For Matt\Directional Surveys\Directional_Survey.csv",
    dtype={"uwi": str})

In [8]:
df_directional.shape

(312046, 12)

### 2.3 Missing DS from Header

In [9]:
missing_uwi = set(df_header['uwi']) - set(df_directional['uwi'])
print(f"Number of missing UWI: {len(missing_uwi)}")
# list(missing_uwi)

Number of missing UWI: 663


## 3. Computing UTM Coordinates

In [10]:
# Initialize the GeoSurveyProcessor with the log directory
geo = GeoSurveyProcessor(
    log_dir=r"C:\Users\Apoorva.Saxena\OneDrive - Sitio Royalties\Desktop\Project - Apoorva\Python\Parent_Child_Spacing\logs")

[GeoLogger] INFO (08-21 09:57 PM): GeoSurveyProcessor initialized. (Line: 477) [well_data_manager.py]



In [11]:
df_utm = geo.compute_utm_coordinates(df=df_directional)

[GeoLogger] INFO (08-21 09:57 PM): ✅ Using lat/lon from input DataFrame. (Line: 653) [well_data_manager.py]

[GeoLogger] INFO (08-21 09:57 PM): ✅ UTM coordinate computation complete in 0.33 sec. (Line: 708) [well_data_manager.py]



In [12]:
# Filter the DataFrame to get only the lateral sections after the heel point
df_utm_lateral = geo.filter_after_heel_point(df=df_utm)

## 4. Calculate Spacing I-K Piars

In [13]:
# Filter the DataFrame based on specific conditions
df_header_filter = df_header[~(df_header["uwi"].isin(missing_uwi)) & 
          (df_header["rsv_cat"].isin(["01PDP", "02PA", "02PDNP", "03PUD"])) &
          ~(df_header["hole_direction"]=="Vertical")].reset_index(drop=True).copy()

In [14]:
list(set(df_header_filter['uwi']) - set(df_utm_lateral[df_utm_lateral["uwi"].isin(df_header_filter["uwi"].unique())]["uwi"]))

[]

In [15]:
# Create an instance of WellSpacingCalculator with the filtered trajectories
spacing_stats_ik = WellSpacingCalculator(trajectories=df_utm_lateral[df_utm_lateral["uwi"].isin(df_header_filter["uwi"].unique())])

In [16]:
# spacing_stats_ik._calculate_spacing_statistics(
#     batch_size=200_000,
#     max_distance_miles=4.0,
#     save_batches_dir=r"C:\Users\Apoorva.Saxena\OneDrive - Sitio Royalties\Desktop\Project - Apoorva\For Matt\Spacing Stats",
#     max_crossline_ft=3000
# )

In [17]:
df_spacing_ik = spacing_stats_ik._load_saved_batches(
    batch_folder=r"C:\Users\Apoorva.Saxena\OneDrive - Sitio Royalties\Desktop\Project - Apoorva\For Matt\Spacing Stats"
)

🔍 Found 2 batch files. Loading and combining...
✅ Loaded 146,840 rows from all batches.


## 5. Get OneLine

### 5.1 Joining bench with spacing ik dataframe

In [18]:
df_spacing_ik_bench = df_spacing_ik.copy()

bench_map = df_header_filter.set_index("uwi")["bench"]

df_spacing_ik_bench["bench_i"] = df_spacing_ik_bench["well_i"].map(bench_map)
df_spacing_ik_bench["bench_k"] = df_spacing_ik_bench["well_k"].map(bench_map)

df_spacing_ik_bench = reorder_columns(df=df_spacing_ik_bench, columns_to_move=['bench_i', 'bench_k'], reference_column='well_k')
df_spacing_ik_bench = reorder_columns(df=df_spacing_ik_bench, 
                                        columns_to_move=['direction_to_k_from_i_axis','overlap_pct_i', 'overlap_pct_k','overlap_len_common_ft', 'LL_i', 'LL_k'], 
                                        reference_column='3D_dist')

### 5.2 Running oneline

In [19]:
# Filter out rows where 'reject_reason' is not empty
df_spacing_ik_bench_filt = df_spacing_ik_bench[df_spacing_ik_bench["reject_reason"]==""].reset_index(drop=True).copy()

In [20]:
nb = DirectionalBenchNeighbors()

# A) Classic: 1320′ cutoff, any axis, no preference
res_any = nb.summarize(spacing_df=df_spacing_ik_bench_filt, header_df=df_header_filter, 
                       cutoff_ft=1800.0, vertical_cutoff_ft=150.00, 
                       overlap_pct_k_min=0.30)   # require ≥30% of k overlaps with i

# B) Only EW is eligible for *_1, and prefer EW if ties occur
res_ew = nb.summarize(spacing_df=df_spacing_ik_bench_filt, header_df=df_header_filter, 
                      cutoff_ft=1800.0, vertical_cutoff_ft=150.00, overlap_pct_k_min=0.30, # require ≥30% of k overlaps with i
                      axis_mode="EW", prefer_axis="EW")

# C) Any axis is eligible for *_1, but prefer NS on distance ties
res_any_pref_ns = nb.summarize(spacing_df=df_spacing_ik_bench_filt, header_df=df_header_filter, 
                               cutoff_ft=1800.0, vertical_cutoff_ft=150.00, overlap_pct_k_min=0.30, # require ≥30% of k overlaps with i
                               axis_mode="any", prefer_axis="NS")

In [22]:
res_ew

,well_i,uwi_same_1,hz_ft_to_same_1,vt_ft_to_same_1,3d_ft_to_same_1,uwi_same_2,hz_ft_to_same_2,vt_ft_to_same_2,3d_ft_to_same_2,uwi_near_1,hz_ft_to_near_1,vt_ft_to_near_1,3d_ft_to_near_1,uwi_near_2,hz_ft_to_near_2,vt_ft_to_near_2,3d_ft_to_near_2
0,30025410040100,30025503690000,933.507723,23.6125,933.806307,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,30025421210000,30025426220000,1107.525915,26.0445,1107.832103,30025428730000,1754.503370,46.6605,1755.123721,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,30025426220000,30025421210000,1107.539330,26.0445,1107.845514,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,30025428730000,30025421210000,1754.494781,46.6605,1755.115136,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,30025437350100,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1859,42501376070000,42501376080000,715.778692,9.7000,715.844415,NaN,NaN,NaN,NaN,42501366580000,850.667092,31.890,851.264632,42501371140000,1466.575518,41.6600,1467.167102
1860,42501376080000,42501376070000,715.777839,9.7000,715.843562,42501376100000,824.248755,7.7850,824.285518,42501371140000,756.582689,51.360,758.323952,42501366580000,1572.672532,41.5900,1573.222369
1861,42501376090000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,42501366970000,906.587087,68.695,909.185981,42501372890000,1700.058754,31.0685,1700.342618
1862,42501376100000,42501376080000,824.177946,7.7850,824.214713,NaN,NaN,NaN,NaN,42501366270000,695.426828,41.965,696.691850,NaN,NaN,NaN,NaN


In [27]:
# debug_pair_spacing(
#     df=df_utm_lateral,
#     uwi_i= "42003437630000",
#     uwi_k="42003417200000"
# )

In [37]:
class SpacingNeighborEnricher:
    """
    Enriches DirectionalBenchNeighbors output with well header attributes and spacing-derived attributes,
    then organizes the columns into a consistent order.

    Inputs
    ------
    header : pd.DataFrame
        Well-level info keyed by 'uwi'. Must contain:
        - 'uwi', 'operator', 'well_name', 'first_prod_date', 'bench'
        (first_prod_date should be datetime or parseable as such)

    spacing : pd.DataFrame
        Pairs-level spacing metrics with columns including:
        - 'well_i', 'well_k'
        - 'drill_direction_i', 'LL_i'            (attributes for well_i)
        - 'drill_direction_k', 'LL_k', 'overlap_pct_k' (attributes for well_k)
        Other columns may be present; only those listed are used here.

        NOTE: Multiple rows per (well_i, well_k) can exist. The first occurrence
        after dropping duplicates is used to define a unique mapping for joins.

    neighbors : pd.DataFrame
        Output of DirectionalBenchNeighbors(). Must contain:
        - 'well_i'
        - zero or more neighbor UWI columns in the form 'uwi_<suffix>' e.g.:
            'uwi_same_1', 'uwi_same_2', 'uwi_near_1', 'uwi_near_2', ...
        - any associated distance columns (e.g., hz_ft_to_same_1) are preserved.

    Behavior
    --------
    - Keeps the same row count and ordering as `neighbors`.
    - Adds base columns on well_i:
        'operator', 'well_name', 'first_prod_dt', 'bench',
        'drill_direction', 'lateral_length_80'
      (the last two are mapped from spacing’s `drill_direction_i`, `LL_i`).

    - For each neighbor suffix discovered from 'uwi_*' columns, adds:
        'operator_<sfx>', 'well_name_<sfx>', 'first_prod_dt_<sfx>', 'bench_<sfx>'
        'drill_direction_<sfx>', 'lateral_length_80_<sfx>'
        'lateral_length_perc_<sfx>', 'month_<sfx>'

        where:
          - lateral_length_perc_<sfx> comes from spacing.overlap_pct_k
            for (well_i, well_k = uwi_<sfx>)
          - month_<sfx> is the signed difference in whole months between
            well_i’s first_prod_dt and neighbor’s first_prod_dt.

    - Columns are reordered consistently:
        1. well_i
        2. Base attributes of well_i
        3. For each suffix in order:
            uwi_<sfx>, distance cols (if present),
            spacing attrs, header attrs
        4. Any remaining columns at the end

    Returns
    -------
    pd.DataFrame
        Same row count/order as `neighbors`, enriched and re-ordered.

    Notes
    -----
    - All operations are vectorized (Series.map or MultiIndex reindex).
    - If spacing lacks a (well_i, well_k) row, added per-suffix fields will be NaN.
    - If spacing contains multiple entries per key, the first occurrence is used.
    """

    def __init__(
        self,
        header: pd.DataFrame,
        spacing: pd.DataFrame,
        neighbors: pd.DataFrame,
        neighbor_prefix: str = "uwi_"
    ) -> None:
        self.header = header.copy()
        self.spacing = spacing.copy()
        self.neighbors = neighbors.copy()
        self.neighbor_prefix = neighbor_prefix

        # Ensure date types
        if not pd.api.types.is_datetime64_any_dtype(self.header.get("first_prod_date")):
            self.header["first_prod_date"] = pd.to_datetime(self.header["first_prod_date"], errors="coerce")

    def _discover_suffixes(self) -> List[str]:
        # Collect all columns that look like 'uwi_<suffix>'
        sfx = []
        for col in self.neighbors.columns:
            if col.startswith(self.neighbor_prefix) and col != "uwi":  # safety if 'uwi' exists
                sfx.append(col[len(self.neighbor_prefix):])
        return sfx

    @staticmethod
    def _months_between(a: pd.Series, b: pd.Series) -> pd.Series:
        """
        Compute signed month difference (b - a) in whole months, handling NaT safely.
        Returns a nullable integer Series (Int64) with <NA> where either side is missing.
        """
        a = pd.to_datetime(a, errors="coerce")
        b = pd.to_datetime(b, errors="coerce")

        ay = a.dt.year.astype("Int64")
        am = a.dt.month.astype("Int64")
        by = b.dt.year.astype("Int64")
        bm = b.dt.month.astype("Int64")

        # Any missing year/month -> result NA
        mask_na = ay.isna() | am.isna() | by.isna() | bm.isna()

        # Month delta ignoring day-of-month
        delta = (by - ay) * 12 + (bm - am)
        delta = delta.astype("Int64")
        delta[mask_na] = pd.NA
        return delta

    def _default_column_order(self, df: pd.DataFrame, suffixes: List[str]) -> List[str]:
        """
        Build a sensible default column order based on discovered suffixes and known patterns.
        Keeps unknown columns at the end.
        """
        cols = []

        # 1) ID
        base_id = ["well_i"]
        cols += [c for c in base_id if c in df.columns]

        # 2) Base attrs for well_i
        base_attrs = ["operator", "well_name", "first_prod_dt", "bench", "drill_direction", "lateral_length_80"]
        cols += [c for c in base_attrs if c in df.columns]

        # 3) Group per suffix in a consistent order
        # Within each suffix, follow: uwi, distances, spacing attrs, header attrs
        for sfx in suffixes:
            uwi_col = f"uwi_{sfx}"
            distances = [f"hz_ft_to_{sfx}", f"vt_ft_to_{sfx}", f"3d_ft_to_{sfx}"]
            spacing_k = [
                f"drill_direction_{sfx}",
                f"lateral_length_80_{sfx}",
                f"lateral_length_perc_{sfx}",
                f"month_{sfx}",
            ]
            header_k = [
                f"operator_{sfx}",
                f"well_name_{sfx}",
                f"first_prod_dt_{sfx}",
                f"bench_{sfx}",
            ]

            for group in [[uwi_col], distances, spacing_k, header_k]:
                cols += [c for c in group if c in df.columns]

        # 4) Add anything we didn't explicitly order at the end
        remainder = [c for c in df.columns if c not in cols]
        return cols + remainder

    def _build_maps(self) -> Dict[str, pd.Series]:
        """
        Build all Series maps needed for vectorized enrichment.
        Returns a dict of Series keyed by a descriptive name.
        """
        maps: Dict[str, pd.Series] = {}

        # --- Header maps keyed by UWI ---
        # We keep names simple for base well_i, and will suffix for neighbors
        header_cols = ["operator", "well_name", "first_prod_date", "bench"]
        for c in header_cols:
            maps[f"header__{c}"] = self.header.set_index("uwi")[c]

        # --- Spacing maps ---
        # For well_i-only attributes, make a unique record per well_i
        # Keep first occurrence after dropping duplicates on well_i
        spacing_i = (
            self.spacing
            .dropna(subset=["well_i"])
            .drop_duplicates(subset=["well_i"])
            .set_index("well_i")
        )
        # These may not exist in all datasets; guard with get
        if "drill_direction_i" in spacing_i.columns:
            maps["spacing_i__drill_direction"] = spacing_i["drill_direction_i"]
        if "LL_i" in spacing_i.columns:
            maps["spacing_i__LL"] = spacing_i["LL_i"]

        # For (well_i, well_k) pair attributes (k-side), make MultiIndex maps
        spacing_pairs = (
            self.spacing
            .dropna(subset=["well_i", "well_k"])
            .drop_duplicates(subset=["well_i", "well_k"])
            .set_index(["well_i", "well_k"])
        )

        for col, key in [
            ("drill_direction_k", "spacing_pair__drill_direction_k"),
            ("LL_k", "spacing_pair__LL_k"),
            ("overlap_pct_k", "spacing_pair__overlap_pct_k"),
        ]:
            if col in spacing_pairs.columns:
                maps[key] = spacing_pairs[col]

        return maps

    def build(self, column_order: Optional[List[str]] = None) -> pd.DataFrame:
        neighbors = self.neighbors.copy()
        suffixes = self._discover_suffixes()
        maps = self._build_maps()

        # Base well_i attributes from header
        uwi_to_operator = maps["header__operator"]
        uwi_to_wellname = maps["header__well_name"]
        uwi_to_fpd      = maps["header__first_prod_date"]
        uwi_to_bench    = maps["header__bench"]

        neighbors["operator"]       = neighbors["well_i"].map(uwi_to_operator)
        neighbors["well_name"]      = neighbors["well_i"].map(uwi_to_wellname)
        neighbors["first_prod_dt"]  = neighbors["well_i"].map(uwi_to_fpd)
        neighbors["bench"]          = neighbors["well_i"].map(uwi_to_bench)

        # From spacing (i-side)
        if "spacing_i__drill_direction" in maps:
            neighbors["drill_direction"] = neighbors["well_i"].map(maps["spacing_i__drill_direction"])
        else:
            neighbors["drill_direction"] = pd.NA

        if "spacing_i__LL" in maps:
            neighbors["lateral_length_80"] = neighbors["well_i"].map(maps["spacing_i__LL"])
        else:
            neighbors["lateral_length_80"] = pd.NA

        # For each neighbor suffix, add header + spacing(k) + months + overlap_pct_k
        # Prepare convenience accessors
        pair_dirk = maps.get("spacing_pair__drill_direction_k")
        pair_llk  = maps.get("spacing_pair__LL_k")
        pair_olpk = maps.get("spacing_pair__overlap_pct_k")

        for sfx in suffixes:
            uwi_col = f"{self.neighbor_prefix}{sfx}"
            if uwi_col not in neighbors.columns:
                continue

            # Neighbor header attributes
            neighbors[f"operator_{sfx}"]      = neighbors[uwi_col].map(uwi_to_operator)
            neighbors[f"well_name_{sfx}"]     = neighbors[uwi_col].map(uwi_to_wellname)
            neighbors[f"first_prod_dt_{sfx}"] = neighbors[uwi_col].map(uwi_to_fpd)
            neighbors[f"bench_{sfx}"]         = neighbors[uwi_col].map(uwi_to_bench)

            # Spacing pair attributes (k-side) via MultiIndex map
            pair_index = pd.MultiIndex.from_frame(
                pd.DataFrame({"well_i": neighbors["well_i"], "well_k": neighbors[uwi_col]})
            )

            if pair_dirk is not None:
                neighbors[f"drill_direction_{sfx}"] = pd.Series(
                    pair_dirk.reindex(pair_index).to_numpy(), index=neighbors.index
                )
            else:
                neighbors[f"drill_direction_{sfx}"] = pd.NA

            if pair_llk is not None:
                neighbors[f"lateral_length_80_{sfx}"] = pd.Series(
                    pair_llk.reindex(pair_index).to_numpy(), index=neighbors.index
                )
            else:
                neighbors[f"lateral_length_80_{sfx}"] = pd.NA

            if pair_olpk is not None:
                neighbors[f"lateral_length_perc_{sfx}"] = pd.Series(
                    pair_olpk.reindex(pair_index).to_numpy(), index=neighbors.index
                )
            else:
                neighbors[f"lateral_length_perc_{sfx}"] = pd.NA

            neighbors[f"month_{sfx}"] = self._months_between(
                neighbors["first_prod_dt"], neighbors[f"first_prod_dt_{sfx}"]
            )

        # Optional: column ordering
        if column_order is not None:
            # Keep existing columns and add any new ones not specified at the end
            ordered = [c for c in column_order if c in neighbors.columns]
            rest = [c for c in neighbors.columns if c not in ordered]
            neighbors = neighbors[ordered + rest]
        else:
            # auto-order using discovered suffixes
            auto_order = self._default_column_order(neighbors, suffixes)
            neighbors = neighbors[auto_order]

        return neighbors

In [38]:
# Initialize the enricher
enricher = SpacingNeighborEnricher(
    header=df_header_filter,
    spacing=df_spacing_ik_bench[df_spacing_ik_bench["reject_reason"]==""],
    neighbors=res_ew
)

# Build the enriched DataFrame
df_enriched = enricher.build()

In [42]:
df_header_filter

,uwi,api10,well_name,operator,first_prod_date,rsv_cat,hole_direction,bench,ihs_missing,ihs_ds_missing
0,42003002890200,4200300289,UNIVERSITY BLOCK 9 4AB,Exxon Mobil,2000-07-15,02PDNP,Horizontal,MISSISSIPPIAN,Not Missing,Not Missing
1,42003029150100,4200302915,UNIVERSITY BLOCK 9 6AO,Exxon Mobil,2011-06-15,02PDNP,Horizontal,SUB-WOODFORD,Missing,Not Missing
2,42003029190100,4200302919,UNIVERSITY BLOCK 9 7AO,Exxon Mobil,2011-09-15,02PDNP,Horizontal,SUB-WOODFORD,Missing,Not Missing
3,42003029190200,4200302919,UNIVERSITY BLOCK 9 7AO,Exxon Mobil,2011-09-15,02PDNP,Horizontal,SUB-WOODFORD,Missing,Not Missing
4,42003029720300,4200302972,UNIVERSITY BLOCK 9 3AT,Exxon Mobil,2001-07-15,01PDP,Horizontal,SUB-WOODFORD,Missing,Not Missing
...,...,...,...,...,...,...,...,...,...,...
1991,42501376070000,4250137607,RUSTY JOHNNY A 604-577 15XH,Riley Permian,NaT,03PUD,Horizontal,WOLFCAMP,Missing,Not Missing
1992,42501376080000,4250137608,RUSTY JOHNNY B 604-577 2XH,Riley Permian,NaT,03PUD,Horizontal,WOLFCAMP,Missing,Not Missing
1993,42501376090000,4250137609,JOSEY WALES A 640-669 8XH,Riley Permian,NaT,03PUD,Horizontal,WOLFCAMP,Missing,Not Missing
1994,42501376100000,4250137610,RUSTY CRANE 604 25H,Riley Permian,NaT,03PUD,Horizontal,WOLFCAMP,Missing,Not Missing


In [39]:
df_enriched

,well_i,operator,well_name,first_prod_dt,bench,drill_direction,lateral_length_80,uwi_same_1,hz_ft_to_same_1,vt_ft_to_same_1,3d_ft_to_same_1,drill_direction_same_1,lateral_length_80_same_1,lateral_length_perc_same_1,month_same_1,operator_same_1,well_name_same_1,first_prod_dt_same_1,bench_same_1,uwi_same_2,hz_ft_to_same_2,vt_ft_to_same_2,3d_ft_to_same_2,drill_direction_same_2,lateral_length_80_same_2,lateral_length_perc_same_2,month_same_2,operator_same_2,well_name_same_2,first_prod_dt_same_2,bench_same_2,uwi_near_1,hz_ft_to_near_1,vt_ft_to_near_1,3d_ft_to_near_1,drill_direction_near_1,lateral_length_80_near_1,lateral_length_perc_near_1,month_near_1,operator_near_1,well_name_near_1,first_prod_dt_near_1,bench_near_1,uwi_near_2,hz_ft_to_near_2,vt_ft_to_near_2,3d_ft_to_near_2,drill_direction_near_2,lateral_length_80_near_2,lateral_length_perc_near_2,month_near_2,operator_near_2,well_name_near_2,first_prod_dt_near_2,bench_near_2
0,30025410040100,Burk Royalty,BROKEN SPOKE 2 STATE #001H,2014-05-02,SAN ANDRES,NS,4193.025870,30025503690000,933.507723,23.6125,933.806307,NS,5300.057789,0.791128,100,Burk Royalty,BROKEN SPOKE 2 STATE #002H,2022-09-20,SAN ANDRES,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,NaN,NaN,NaT,NaN
1,30025421210000,Burk Royalty,DOG BAR 11 FEE #002H,2015-03-01,SAN ANDRES,NS,4305.683334,30025426220000,1107.525915,26.0445,1107.832103,NS,4319.159191,0.996295,6,Burk Royalty,DOG BAR 11 FEE #003H,2015-09-01,SAN ANDRES,30025428730000,1754.503370,46.6605,1755.123721,NS,4208.582903,0.999952,10,Burk Royalty,DOG BAR 11 FEE #001H,2016-01-01,SAN ANDRES,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,NaN,NaN,NaT,NaN
2,30025426220000,Burk Royalty,DOG BAR 11 FEE #003H,2015-09-01,SAN ANDRES,NS,4319.159191,30025421210000,1107.539330,26.0445,1107.845514,NS,4305.683334,0.989835,-6,Burk Royalty,DOG BAR 11 FEE #002H,2015-03-01,SAN ANDRES,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,NaN,NaN,NaT,NaN
3,30025428730000,Burk Royalty,DOG BAR 11 FEE #001H,2016-01-01,SAN ANDRES,NS,4208.582903,30025421210000,1754.494781,46.6605,1755.115136,NS,4305.683334,0.974967,-10,Burk Royalty,DOG BAR 11 FEE #002H,2015-03-01,SAN ANDRES,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,NaN,NaN,NaT,NaN
4,30025437350100,Burk Royalty,POLLOS HERMANOS STATE COM #005H,2017-07-13,SAN ANDRES,NS,6984.627251,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,NaN,NaN,NaT,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1859,42501376070000,Riley Permian,RUSTY JOHNNY A 604-577 15XH,NaT,WOLFCAMP,NS,10398.014964,42501376080000,715.778692,9.7000,715.844415,NS,10542.588119,0.986227,<NA>,Riley Permian,RUSTY JOHNNY B 604-577 2XH,NaT,WOLFCAMP,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,NaN,NaN,NaT,NaN,42501366580000,850.667092,31.890,851.264632,NS,7200.567884,0.999516,<NA>,Riley Permian,RUSTY CRANE 604-577 1XH,2016-11-15,SAN ANDRES,42501371140000,1466.575518,41.6600,1467.167102,NS,4868.685192,0.998207,<NA>,Riley Permian,JOHNNY TYLER 577 3H,2021-01-15,SAN ANDRES
1860,42501376080000,Riley Permian,RUSTY JOHNNY B 604-577 2XH,NaT,WOLFCAMP,NS,10542.588119,42501376070000,715.777839,9.7000,715.843562,NS,10398.014964,0.999941,<NA>,Riley Permian,RUSTY JOHNNY A 604-577 15XH,NaT,WOLFCAMP,42501376100000,824.248755,7.7850,824.285518,NS,5309.054650,0.985639,<NA>,Riley Permian,RUSTY CRANE 604 25H,NaT,WOLFCAMP,42501371140000,756.582689,51.360,758.323952,NS,4868.685192,0.998249,<NA>,Riley Permian,JOHNNY TYLER 577 3H,2021-01-15,SA

In [ ]:
res_ew

,well_i,uwi_same_1,hz_ft_to_same_1,vt_ft_to_same_1,3d_ft_to_same_1,uwi_same_2,hz_ft_to_same_2,vt_ft_to_same_2,3d_ft_to_same_2,uwi_near_1,hz_ft_to_near_1,vt_ft_to_near_1,3d_ft_to_near_1,uwi_near_2,hz_ft_to_near_2,vt_ft_to_near_2,3d_ft_to_near_2
0,30025410040100,30025503690000,933.507723,23.6125,933.806307,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,30025421210000,30025426220000,1107.525915,26.0445,1107.832103,30025428730000,1754.503370,46.6605,1755.123721,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,30025426220000,30025421210000,1107.539330,26.0445,1107.845514,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,30025428730000,30025421210000,1754.494781,46.6605,1755.115136,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,30025437350100,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1859,42501376070000,42501376080000,715.778692,9.7000,715.844415,NaN,NaN,NaN,NaN,42501366580000,850.667092,31.890,851.264632,42501371140000,1466.575518,41.6600,1467.167102
1860,42501376080000,42501376070000,715.777839,9.7000,715.843562,42501376100000,824.248755,7.7850,824.285518,42501371140000,756.582689,51.360,758.323952,42501366580000,1572.672532,41.5900,1573.222369
1861,42501376090000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,42501366970000,906.587087,68.695,909.185981,42501372890000,1700.058754,31.0685,1700.342618
1862,42501376100000,42501376080000,824.177946,7.7850,824.214713,NaN,NaN,NaN,NaN,42501366270000,695.426828,41.965,696.691850,NaN,NaN,NaN,NaN


In [41]:
df_spacing_ik_bench_filt

,well_i,well_k,bench_i,bench_k,horizontal_dist,horizontal_dist_median,vertical_dist,3D_dist,direction_to_k_from_i_axis,overlap_pct_i,overlap_pct_k,overlap_len_common_ft,LL_i,LL_k,drill_direction_i,drill_direction_k,n_samples,dy_p5,angle_deg,pair_alignment,min_distance_ft,mean_windowed_ft,reject_reason,direction_axis,direction_axis_confidence,direction_axis_distribution,axis_forced
0,30025410040100,30025455300000,SAN ANDRES,SAN ANDRES,2834.430974,2836.702875,29.4475,2834.583938,W,0.966272,0.572607,4051.602847,4193.025870,7075.715781,NS,NS,41.0,2803.409515,0.336438,parallel_like,NaN,NaN,,EW,1.0,"E:0.00,W:1.00",True
1,30025410040100,30025503690000,SAN ANDRES,SAN ANDRES,933.507723,940.956180,23.6125,933.806307,W,1.000000,0.791128,4193.025870,4193.025870,5300.057789,NS,NS,42.0,889.457666,1.111058,parallel_like,NaN,NaN,,EW,1.0,"E:0.00,W:1.00",True
2,30025410040100,42501364850000,SAN ANDRES,SAN ANDRES,2870.944865,2849.950825,4.1775,2870.947904,E,0.794858,0.783564,3332.859431,4193.025870,4253.461365,NS,NS,34.0,2696.434016,5.870203,parallel_like,NaN,NaN,,EW,1.0,"E:1.00,W:0.00",True
3,30025410040100,42501372540000,SAN ANDRES,SAN ANDRES,1505.026059,1505.095305,27.3745,1505.274992,E,0.061805,0.051191,259.151954,4193.025870,5062.409197,NS,NS,3.0,1500.649478,1.592600,parallel_like,NaN,NaN,,EW,1.0,"E:1.00,W:0.00",True
4,30025410040100,42501373440000,SAN ANDRES,SAN ANDRES,2918.138197,2920.880702,29.2850,2918.285139,E,0.088816,0.070390,372.409695,4193.025870,5290.682747,NS,NS,4.0,2812.767973,7.202972,parallel_like,NaN,NaN,,EW,1.0,"E:1.00,W:0.00",True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15743,42501376110000,42501373040000,SAN ANDRES,SAN ANDRES,668.991547,677.768394,61.9835,671.856863,E,0.964660,0.998868,4982.741352,5165.280109,4988.388101,NS,NS,50.0,546.092506,2.734059,parallel_like,NaN,NaN,,EW,1.0,"E:1.00,W:0.00",True
15744,42501376110000,42501373050000,SAN ANDRES,SAN ANDRES,834.861571,823.656159,46.1885,836.138278,W,0.978777,0.998863,5055.658011,5165.280109,5061.413897,NS,NS,51.0,713.693706,2.580199,parallel_like,NaN,NaN,,EW,1.0,"E:0.00,W:1.00",True
15745,42501376110000,42501375350000,SAN ANDRES,WOLFCAMP,2093.235707,2095.556100,1.7250,2093.236418,W,0.954892,0.635866,4932.283826,5165.280109,7756.799243,NS,NS,50.0,2039.530034,0.628341,parallel_like,NaN,NaN,,EW,1.0,"E:0.00,W:1.00",True
15746,42501376110000,42501375360000,SAN ANDRES,WOLFCAMP,2761.563591,2766.038907,25.3250,2761.679711,W,0.995837,0.953752,5143.779023,5165.280109,5393.206482,NS,NS,52.0,2711.309274,0.512467,parallel_like,NaN,NaN,,EW,1.0,"E:0.00,W:1.00",True
